## Emergence of the profile during fine-tuning

The reverb-competent MUSE checkpoint is the endpoint of a 50-epoch fine-tuning run, and the artifacts include CKA sweeps at ten intermediate checkpoints (epochs 1, 3, 5, 7, 9, 10, 12, 24, 36 and 48) on the noise axis, 824 utterances × 41 SNRs × five noises. Refitting the per-layer line at each checkpoint lets us watch the profile form (Sec. "Origin of the Profile in Training", second part): the organisation is present at the first probed checkpoint and only sharpens, and the change fine-tuning induces concentrates in exactly the stages the profile labels as sensitive. This chapter reproduces those numbers from the shipped per-epoch profiles with `se_probe.profiles.emergence_convergence`.

Runtime: under 20 seconds on any device.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


def _rel(p):
    """Repo-relative path for display (absolute only if outside the repo)."""
    try:
        return p.relative_to(REPO)
    except ValueError:
        return p

from se_probe.device import device_info, get_device
from se_probe.plotting import apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import numpy as np
import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
TABLES = REPO / "results_tables"
print(f"Using data from: {_rel(DATA_DIR)}")
print(f"Paper tables from: {_rel(TABLES)}")

In [ ]:
import json

from se_probe.layers import MUSE_STAGE_LABELS, layout, muse_stage_of, probed_layers
from se_probe.profiles import emergence_convergence

EPOCHS = [1, 3, 5, 7, 9, 10, 12, 24, 36, 48]
t = pd.read_csv(TABLES / "emergence/IIIF_per_layer_profiles_new5.csv")
assert list(t["layer"]) == probed_layers("muse", t["layer"]), "rows are not in depth order"

profiles = {"pretrained": pd.DataFrame({"layer": t["layer"], "alpha": t["alpha_pre"], "beta": t["beta_pre"]})}
for e in EPOCHS:
    profiles[e] = pd.DataFrame({"layer": t["layer"], "alpha": t[f"alpha_e{e}"], "beta": t[f"beta_e{e}"]})

res = emergence_convergence(profiles, final_key=48, pretrained_key="pretrained")
conv = res["convergence"].set_index("epoch")
conc = res["concentration"]
print(conv.round(4))

In [ ]:
ref = {e["label"]: e for e in json.load(open(TABLES / "emergence/IIIF_emergence.json"))}
e1, e36 = conv.loc[1, "beta_r_vs_final"], conv.loc[36, "beta_r_vs_final"]
print(f"beta profile r with epoch 48: epoch 1 {e1:.2f} (paper 0.76) -> epoch 36 {e36:.4f} (paper >= 0.99)")
print(f"pretrained checkpoint vs epoch 48: beta r {conc['pretrained_beta_r_vs_final']:.2f} (paper 0.76), alpha r {conc['pretrained_alpha_r_vs_final']:.2f} (paper 0.73)")
assert abs(e1 - 0.76) < 5e-3 and e36 >= 0.99 and abs(conv.loc[48, "beta_r_vs_final"] - 1.0) < 1e-9
assert abs(conc["pretrained_beta_r_vs_final"] - 0.76) < 5e-3 and abs(conc["pretrained_alpha_r_vs_final"] - 0.73) < 5e-3
assert abs(e1 - ref["new5"]["headline"]["e1_beta_r"]) < 1e-9

rb, ra = conc["ratio_dbeta_sensitive_over_reference"], conc["ratio_dalpha_sensitive_over_reference"]
print(f"mean |delta beta| decoder+refinement / Enc-L1 = {rb:.2f}x   (shipped new5 3.40, old5 {ref['old5']['concentration']['ratio_dbeta_decref_over_encl1']:.2f}; paper quotes 3x)")
print(f"mean |delta alpha| decoder+refinement / Enc-L1 = {ra:.2f}x   (shipped new5 5.77, old5 {ref['old5']['concentration']['ratio_dalpha_decref_over_encl1']:.2f}; paper quotes 5.8x)")
assert abs(rb - 3.40) < 0.05 and abs(ra - 5.77) < 0.05
assert abs(rb - ref["new5"]["concentration"]["ratio_dbeta_decref_over_encl1"]) < 1e-9

The manuscript rounds the concentration ratios to "3×" and "5.8×"; the shipped refit on the five test-split noises gives 3.40 and 5.77 (3.06 and 5.50 on the earlier PCAFETER five). Both files are in `results_tables/emergence/`.

In [ ]:
import matplotlib.pyplot as plt

from se_probe.plotting import OKABE_ITO, depth_axis

lay = layout("muse")
x = np.arange(24)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), gridspec_kw={"width_ratios": [2.2, 1.2, 1.2]})

ax = axes[0]
depth_axis(ax, lay)
show = ["pretrained", 1, 5, 12, 48]
cmap = plt.get_cmap("viridis")
for i, k in enumerate(show):
    ax.plot(x, profiles[k]["beta"], marker="o", ms=2.8, lw=1.1, color="0.4" if k == "pretrained" else cmap(i / (len(show) - 1)),
            ls="--" if k == "pretrained" else "-", label=k if k == "pretrained" else f"epoch {k}")
ax.set_ylabel(r"$\beta_\ell$ (CKA/dB)")
ax.set_title("(a) slope profile along fine-tuning")
ax.legend(fontsize=7, ncol=2)

ax = axes[1]
ep = conv.drop(index="pretrained")
ax.plot(ep.index.astype(int), ep["beta_r_vs_final"], "o-", color=OKABE_ITO["blue"], ms=4, label=r"$\beta_\ell$")
ax.plot(ep.index.astype(int), ep["alpha_r_vs_final"], "s--", color=OKABE_ITO["vermillion"], ms=4, label=r"$\alpha_\ell$")
ax.axhline(conc["pretrained_beta_r_vs_final"], color="0.5", ls=":", lw=1, label="pretrained")
ax.set_xscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("Pearson r with epoch 48")
ax.set_title("(b) convergence")
ax.legend(fontsize=7)

ax = axes[2]
stages = [muse_stage_of(L) for L in t["layer"]]
db = np.abs(profiles["pretrained"]["beta"].to_numpy() - profiles[48]["beta"].to_numpy())
da = np.abs(profiles["pretrained"]["alpha"].to_numpy() - profiles[48]["alpha"].to_numpy())
st = pd.DataFrame({"stage": stages, "dbeta": db, "dalpha": da}).groupby("stage", sort=False).mean()
w = 0.38
xs = np.arange(len(st))
ax.bar(xs - w / 2, st["dbeta"] / st["dbeta"].iloc[0], w, color=OKABE_ITO["blue"], label=r"$|\Delta\beta|$")
ax.bar(xs + w / 2, st["dalpha"] / st["dalpha"].iloc[0], w, color=OKABE_ITO["vermillion"], label=r"$|\Delta\alpha|$")
ax.set_xticks(xs)
ax.set_xticklabels([MUSE_STAGE_LABELS[s] for s in st.index], rotation=30, ha="right")
ax.set_ylabel("mean |change| relative to Enc-L1")
ax.set_title("(c) pretrained -> epoch 48")
ax.legend(fontsize=7)
plt.tight_layout();